# Module 1.2 — Retrieval Metrics: LLM-Judged

Module 1.1's metrics all need pre-labeled ground truth — a `relevant_ids` list, written by hand, per query. That doesn't scale to arbitrary production traffic. DeepEval's three **Contextual** metrics solve the same "is retrieval working" question using an LLM judge instead of hand-labeled IDs, at the cost of being LLM-judged (see Module 0 §3 for the general caveats that come with that).

_Source: adapted from `RAG_Evaluation/1.Retriever_Evaluation_Metrics.ipynb` and condensed from `RAG_Evaluation/DeepEval_Metrics/Contextual_{Precision,Recall,Relevancy}.ipynb` (each of those drilled the same metric across 8-10 near-identical examples; this notebook keeps the 2-3 per metric that teach something distinct). The originals build their examples against a live RAG pipeline (`%run Build_RAG_Pipeline_with_Source.ipynb`); this notebook uses hand-constructed `retrieval_context` instead, so it runs standalone — the live-pipeline version of this exact evaluation is Module 1.7's capstone.

## The three metrics, and how they differ

All three take a `retrieval_context` (the chunks your retriever actually returned) and judge it against the query and/or a reference answer — but they check for **different failure modes**, which is exactly why DeepEval ships all three rather than one general "is retrieval good" score:

| Metric | Question | Needs `expected_output`? | Catches |
|---|---|---|---|
| **Contextual Precision** | Are the *relevant* chunks ranked *higher* than the irrelevant ones? | Yes | A good chunk buried below noise — same failure Module 1.1's Precision@K catches, but without hand-labeled IDs |
| **Contextual Recall** | Does the retrieved context, as a whole, cover everything the expected answer needs? | Yes | Missing information — the retriever found *something*, but not *enough* |
| **Contextual Relevancy** | Of everything retrieved, how much is actually on-topic? | No | Noise/off-topic chunks diluting the context, independent of ranking or completeness |

Precision cares about **order**, Recall cares about **coverage**, Relevancy cares about **signal-to-noise** — a retriever can fail any one of these while passing the other two.

### Contextual Precision — does ranking put relevant chunks first?

In [ ]:
# ============ CONTEXTUAL PRECISION ============
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric

query = "What is Apache Spark used for?"
expected_output = "Apache Spark is used for big data processing, real-time analytics, and machine learning."

# Scenario A: good ranking -- the two relevant chunks come first
good_ranking = LLMTestCase(
    input=query,
    actual_output="Apache Spark is a unified analytics engine used for large-scale data processing and real-time analytics.",
    expected_output=expected_output,
    retrieval_context=[
        "Apache Spark is a unified analytics engine for large-scale data processing.",  # relevant
        "Spark is widely used for real-time stream processing and analytics.",  # relevant
        "Spark was originally developed at UC Berkeley's AMPLab in 2009.",  # off-topic trivia
    ],
)

# Scenario B: same chunks, bad ranking -- the off-topic trivia comes first
bad_ranking = LLMTestCase(
    input=query,
    actual_output="Apache Spark is a unified analytics engine used for large-scale data processing and real-time analytics.",
    expected_output=expected_output,
    retrieval_context=[
        "Spark was originally developed at UC Berkeley's AMPLab in 2009.",  # off-topic trivia, now first
        "Apache Spark is a unified analytics engine for large-scale data processing.",  # relevant
        "Spark is widely used for real-time stream processing and analytics.",  # relevant
    ],
)

precision_metric = ContextualPrecisionMetric(threshold=0.5, model="gpt-4o", include_reason=True)

for label, case in [("Good ranking", good_ranking), ("Bad ranking (same chunks, reordered)", bad_ranking)]:
    precision_metric.measure(case)
    print(f"{label}: score={precision_metric.score:.2f}  reason={precision_metric.reason}\n")

**Reading the output:** same three chunks, same relevant/irrelevant split — only the *order* changed. The bad-ranking case should score meaningfully lower, because Contextual Precision specifically rewards relevant chunks ranking above irrelevant ones, not just being present somewhere in the context.

### Contextual Recall — does retrieval cover everything the answer needs?

In [ ]:
# ============ CONTEXTUAL RECALL ============
from deepeval.metrics import ContextualRecallMetric

query = "What are the symptoms of Type 2 Diabetes?"
expected_output = (
    "Type 2 Diabetes presents with symptoms including increased thirst, frequent urination, "
    "unexplained weight loss, fatigue, blurred vision, and slow-healing wounds."
)

# Scenario A: complete coverage -- context contains every symptom the expected answer mentions
complete_context = LLMTestCase(
    input=query,
    actual_output="Symptoms include increased thirst, frequent urination, fatigue, blurred vision, and slow-healing wounds.",
    expected_output=expected_output,
    retrieval_context=[
        "Type 2 Diabetes commonly causes increased thirst and frequent urination.",
        "Fatigue, blurred vision, and unexplained weight loss are also common symptoms.",
        "Wounds and infections may heal more slowly in people with Type 2 Diabetes.",
    ],
)

# Scenario B: partial coverage -- context is missing several symptoms the expected answer needs
partial_context = LLMTestCase(
    input=query,
    actual_output="Symptoms include increased thirst and frequent urination.",
    expected_output=expected_output,
    retrieval_context=[
        "Type 2 Diabetes commonly causes increased thirst and frequent urination.",
    ],
)

recall_metric = ContextualRecallMetric(threshold=0.5, model="gpt-4o", include_reason=True)

for label, case in [("Complete context", complete_context), ("Partial context (missing symptoms)", partial_context)]:
    recall_metric.measure(case)
    print(f"{label}: score={recall_metric.score:.2f}  reason={recall_metric.reason}\n")

**Reading the output:** Contextual Recall checks the retrieved context against `expected_output`, not `actual_output` — it's asking "did the retriever fetch enough to *support* the full expected answer," independent of whether the generator actually used all of it. The partial-context case should score lower because several symptoms named in `expected_output` (weight loss, fatigue, blurred vision, slow-healing wounds) have no supporting chunk at all.

### Contextual Relevancy — how much of what was retrieved is actually on-topic?

In [ ]:
# ============ CONTEXTUAL RELEVANCY ============
from deepeval.metrics import ContextualRelevancyMetric

query = "How does climate change affect ocean levels?"

# Scenario A: high relevancy -- every chunk is on-topic
high_relevancy = LLMTestCase(
    input=query,
    actual_output="Ocean levels rise mainly due to thermal expansion of warming water and melting land ice.",
    retrieval_context=[
        "Rising global temperatures cause ocean water to expand, a phenomenon known as thermal expansion.",
        "Melting glaciers and ice sheets add additional water volume to the oceans.",
    ],
)

# Scenario B: low relevancy -- mostly noise, one on-topic chunk buried in it
low_relevancy = LLMTestCase(
    input=query,
    actual_output="Ocean levels rise mainly due to thermal expansion of warming water and melting land ice.",
    retrieval_context=[
        "The Pacific Ocean is the largest and deepest of Earth's oceans.",
        "Coral reefs support roughly a quarter of all marine species.",
        "Rising global temperatures cause ocean water to expand, a phenomenon known as thermal expansion.",
        "Ocean currents are driven by wind, water density, and tides.",
    ],
)

relevancy_metric = ContextualRelevancyMetric(threshold=0.5, model="gpt-4o", include_reason=True)

for label, case in [("High relevancy (all on-topic)", high_relevancy), ("Low relevancy (mostly noise)", low_relevancy)]:
    relevancy_metric.measure(case)
    print(f"{label}: score={relevancy_metric.score:.2f}  reason={relevancy_metric.reason}\n")

**Reading the output:** unlike Precision (which needs `expected_output` and cares about *order*) and Recall (which needs `expected_output` and cares about *coverage*), Relevancy needs neither ground truth nor cares about order — it's the simplest of the three, purely a signal-to-noise check on `retrieval_context` against `input`. The low-relevancy case should score lower even though the one relevant chunk is present, because 3 of the 4 chunks are noise.

## Summary

- **Contextual Precision** = is the ranking good (relevant chunks first)? *(needs `expected_output`)*
- **Contextual Recall** = is the coverage complete (nothing the answer needs is missing)? *(needs `expected_output`)*
- **Contextual Relevancy** = is the signal-to-noise ratio good (little irrelevant clutter)? *(referenceless)*
- All three are LLM-judged — keep Module 0 §3's caveats (judge-model choice, non-determinism, cost) in mind when reading a single score as gospel.
- Next: [Module 1.3](03_Generator_Metrics_Referenceless.ipynb) moves past retrieval to the generation step — given what was retrieved, is the *answer* any good?